# Tiền xử lý dữ liệu chuyến bay và thời tiết (NB, DN, TSN)
Notebook này thực hiện đầy đủ các bước tiền xử lý cho bài toán dự đoán delay:
- Nạp dữ liệu raw từ 3 sân bay và weather
- Chuẩn hóa schema, thời gian, trạng thái
- Tạo nhãn delay và gom snapshot
- Ghép weather theo thời gian gần nhất trước thời điểm crawl
- Tạo feature và xuất dữ liệu processed

In [51]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
DELAY_THRESHOLD_MINUTES = 15
TRAINING_SNAPSHOT_LEAD_MINUTES = 30

## Bước 1 - Nạp dữ liệu raw
Bước này tự động tìm thư mục raw và nạp 4 file: NB.csv, DN.csv, TSN.csv, weather.csv.
Mục tiêu là tạo bảng flights_raw gồm dữ liệu hợp nhất của 3 sân bay.

In [52]:
def find_raw_dir() -> Path:
    candidates = [
        Path.cwd() / "Data Collection + Processing" / "data" / "raw",
        Path.cwd().parent / "Data Collection + Processing" / "data" / "raw",
        Path.cwd().parent.parent / "Data Collection + Processing" / "data" / "raw",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Khong tim thay thu muc Data Collection + Processing/data/raw")


def load_flight_file(path: Path, source_airport: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["Source Airport"] = source_airport
    return df


raw_dir = find_raw_dir()
processed_dir = raw_dir.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

flight_files = {
    "NB": raw_dir / "NB.csv",
    "DN": raw_dir / "DN.csv",
    "TSN": raw_dir / "TSN.csv",
}

flight_frames = []
for airport_code, file_path in flight_files.items():
    if file_path.exists():
        flight_frames.append(load_flight_file(file_path, airport_code))
    else:
        print(f"Canh bao: khong tim thay {file_path}")

if not flight_frames:
    raise ValueError("Khong co file flight nao de xu ly")

flights_raw = pd.concat(flight_frames, ignore_index=True)
weather_raw = pd.read_csv(raw_dir / "weather.csv", encoding="utf-8-sig")

print("Raw dir:", raw_dir)
print("Flights raw shape:", flights_raw.shape)
print("Weather raw shape:", weather_raw.shape)
flights_raw.head(3)

Raw dir: e:\MyProject\data-monitoring-lab\Data Collection + Processing\data\raw
Flights raw shape: (2203, 9)
Weather raw shape: (544, 10)


,Data Retrieved At (VN),Flight Date,Direction,Scheduled Time,Estimated Time,Airport,Flight Number,Status,Source Airport
0,2026-04-09 23:40:13,2026-04-09,Arrival,00:10,00:07,HO CHI MINH,9G886,Arrived,NB
1,2026-04-09 23:40:13,2026-04-09,Arrival,00:25,00:32,HO CHI MINH,VJ1128,Arrived,NB
2,2026-04-09 23:40:13,2026-04-09,Arrival,07:10,07:00,HO CHI MINH,BL6002,Arrived,NB


## Bước 2 - Chuẩn hóa cột và giá trị text
Bước này chuẩn hóa tên cột về dạng snake_case, làm sạch chuỗi, đổi tên các cột theo schema thống nhất, và map trạng thái chuyến bay về nhóm để tạo nhãn.

In [61]:
def normalize_space(x):
    if pd.isna(x):
        return np.nan
    return re.sub(r"\s+", " ", str(x)).strip()


def normalize_flight_number(x):
    if pd.isna(x):
        return np.nan
    s = normalize_space(x).upper().replace(" ", "")
    s = s.replace("-", "")
    return s


def normalize_route_airport(x):
    if pd.isna(x):
        return np.nan
    s = normalize_space(x).upper()
    accent_map = str.maketrans({
        "À": "A", "Á": "A", "Ả": "A", "Ã": "A", "Ạ": "A",
        "Ă": "A", "Ằ": "A", "Ắ": "A", "Ẳ": "A", "Ẵ": "A", "Ặ": "A",
        "Â": "A", "Ầ": "A", "Ấ": "A", "Ẩ": "A", "Ẫ": "A", "Ậ": "A",
        "Đ": "D",
        "È": "E", "É": "E", "Ẻ": "E", "Ẽ": "E", "Ẹ": "E",
        "Ê": "E", "Ề": "E", "Ế": "E", "Ể": "E", "Ễ": "E", "Ệ": "E",
        "Ì": "I", "Í": "I", "Ỉ": "I", "Ĩ": "I", "Ị": "I",
        "Ò": "O", "Ó": "O", "Ỏ": "O", "Õ": "O", "Ọ": "O",
        "Ô": "O", "Ồ": "O", "Ố": "O", "Ổ": "O", "Ỗ": "O", "Ộ": "O",
        "Ơ": "O", "Ờ": "O", "Ớ": "O", "Ở": "O", "Ỡ": "O", "Ợ": "O",
        "Ù": "U", "Ú": "U", "Ủ": "U", "Ũ": "U", "Ụ": "U",
        "Ư": "U", "Ừ": "U", "Ứ": "U", "Ử": "U", "Ữ": "U", "Ự": "U",
        "Ỳ": "Y", "Ý": "Y", "Ỷ": "Y", "Ỹ": "Y", "Ỵ": "Y",
    })
    s = s.translate(accent_map)
    s = re.sub(r"\s+", " ", s).strip()
    replacements = {
        "TP. HO CHI MINH": "HO CHI MINH",
        "HO CHI MINH": "HO CHI MINH",
        "HA NOI": "HA NOI",
        "DA NANG": "DA NANG",
        "BUON MA THUOT": "BUON MA THUOT",
        "CAN THO": "CAN THO",
        "HAI PHONG": "HAI PHONG",
        "THANH HOA": "THANH HOA",
        "PHU QUOC": "PHU QUOC",
        "CON DAO": "CON DAO",
        "QUY NHON": "QUY NHON",
        "HUE": "HUE",
        "NHA TRANG": "NHA TRANG",
        "PLEIKU": "PLEIKU",
        "CHU LAI": "CHU LAI",
        "CAM RANH": "CAM RANH",
        "VINH": "VINH",
    }
    for old_text, new_text in replacements.items():
        s = s.replace(old_text, new_text)
    s = re.sub(r"\s*\(([^)]+)\)", lambda m: f" ({m.group(1).upper()})", s)
    return s


status_map = {
    "da ha canh": "landed",
    "arrived": "landed",
    "landed": "landed",
    "den tre": "delayed",
    "cham": "delayed",
    "delayed": "delayed",
    "dang bay": "enroute",
    "on time": "on_time",
    "dung gio": "on_time",
    "khoi hanh": "departed",
    "departed": "departed",
    "huy": "cancelled",
    "cancelled": "cancelled",
}


def normalize_status(status_text):
    s = normalize_space(status_text)
    if pd.isna(s) or s == "":
        return "unknown"
    s_ascii = (
        s.lower()
        .replace("đ", "d")
        .replace("á", "a").replace("à", "a").replace("ả", "a").replace("ã", "a").replace("ạ", "a")
        .replace("ă", "a").replace("ắ", "a").replace("ằ", "a").replace("ẳ", "a").replace("ẵ", "a").replace("ặ", "a")
        .replace("â", "a").replace("ấ", "a").replace("ầ", "a").replace("ẩ", "a").replace("ẫ", "a").replace("ậ", "a")
        .replace("é", "e").replace("è", "e").replace("ẻ", "e").replace("ẽ", "e").replace("ẹ", "e")
        .replace("ê", "e").replace("ế", "e").replace("ề", "e").replace("ể", "e").replace("ễ", "e").replace("ệ", "e")
        .replace("í", "i").replace("ì", "i").replace("ỉ", "i").replace("ĩ", "i").replace("ị", "i")
        .replace("ó", "o").replace("ò", "o").replace("ỏ", "o").replace("õ", "o").replace("ọ", "o")
        .replace("ô", "o").replace("ố", "o").replace("ồ", "o").replace("ổ", "o").replace("ỗ", "o").replace("ộ", "o")
        .replace("ơ", "o").replace("ớ", "o").replace("ờ", "o").replace("ở", "o").replace("ỡ", "o").replace("ợ", "o")
        .replace("ú", "u").replace("ù", "u").replace("ủ", "u").replace("ũ", "u").replace("ụ", "u")
        .replace("ư", "u").replace("ứ", "u").replace("ừ", "u").replace("ử", "u").replace("ữ", "u").replace("ự", "u")
        .replace("ý", "y").replace("ỳ", "y").replace("ỷ", "y").replace("ỹ", "y").replace("ỵ", "y")
    )

    for k, v in status_map.items():
        if k in s_ascii:
            return v
    return "other"


col_rename = {
    "Data Retrieved At (VN)": "retrieved_at_vn",
    "Flight Date": "flight_date",
    "Direction": "direction",
    "Scheduled Time": "scheduled_time",
    "Estimated Time": "estimated_time",
    "Airport": "route_airport",
    "Flight Number": "flight_number",
    "Status": "status_raw",
    "Source Airport": "source_airport",
}

flights = flights_raw.rename(columns=col_rename).copy()

for c in ["source_airport", "direction", "route_airport", "status_raw", "flight_number", "scheduled_time", "estimated_time"]:
    flights[c] = flights[c].apply(normalize_space)

flights["flight_number"] = flights["flight_number"].apply(normalize_flight_number)
flights["direction"] = flights["direction"].str.title()
flights["route_airport_std"] = flights["route_airport"].apply(normalize_route_airport)
flights["status_group"] = flights["status_raw"].apply(normalize_status)

flights[["source_airport", "direction", "route_airport", "route_airport_std", "flight_number", "status_raw", "status_group"]].head(5)

,source_airport,direction,route_airport,route_airport_std,flight_number,status_raw,status_group
0,NB,Arrival,HO CHI MINH,HO CHI MINH,9G886,Arrived,landed
1,NB,Arrival,HO CHI MINH,HO CHI MINH,VJ1128,Arrived,landed
2,NB,Arrival,HO CHI MINH,HO CHI MINH,BL6002,Arrived,landed
3,NB,Arrival,DA NANG,DA NANG,VN156,Arrived,landed
4,NB,Arrival,HO CHI MINH,HO CHI MINH,VU750,Arrived,landed


## Bước 3 - Chuẩn hóa datetime và tạo delay_minutes
Bước này parse cột ngày giờ thành datetime, xử lý trường hợp qua ngày (giờ dự kiến < giờ lịch), sau đó tính delay_minutes và tạo nhãn label_delay.

In [63]:
def parse_hhmm(s):
    s = normalize_space(s)
    if pd.isna(s) or s == "":
        return pd.NaT
    dt = pd.to_datetime(s, format="%H:%M", errors="coerce")
    return dt


flights["retrieved_at_vn"] = pd.to_datetime(flights["retrieved_at_vn"], errors="coerce")
flights["flight_date"] = pd.to_datetime(flights["flight_date"], errors="coerce").dt.date

sched_parsed = flights["scheduled_time"].apply(parse_hhmm)
est_parsed = flights["estimated_time"].apply(parse_hhmm)

flights["scheduled_dt"] = pd.to_datetime(flights["flight_date"].astype(str) + " " + sched_parsed.dt.strftime("%H:%M"), errors="coerce")
flights["estimated_dt"] = pd.to_datetime(flights["flight_date"].astype(str) + " " + est_parsed.dt.strftime("%H:%M"), errors="coerce")

# Chi coi la qua ngay neu chuyến khuya va gio du kien nam sang som hom sau.
sched_hour = flights["scheduled_dt"].dt.hour
est_hour = flights["estimated_dt"].dt.hour
cross_day_mask = (
    flights["estimated_dt"].notna()
    & flights["scheduled_dt"].notna()
    & (flights["estimated_dt"] < flights["scheduled_dt"])
    & (sched_hour >= 18)
    & (est_hour <= 6)
)
flights.loc[cross_day_mask, "estimated_dt"] = flights.loc[cross_day_mask, "estimated_dt"] + pd.Timedelta(days=1)

flights["delay_minutes"] = (flights["estimated_dt"] - flights["scheduled_dt"]).dt.total_seconds() / 60.0

# Cat nguong delay bat thuong de giam anh huong du lieu loi.
flights.loc[flights["delay_minutes"].abs() > 600, "delay_minutes"] = np.nan

flights["label_delay"] = np.where(
    flights["delay_minutes"].notna(),
    (flights["delay_minutes"] >= DELAY_THRESHOLD_MINUTES).astype(int),
    np.nan,
)

status_delay_mask = flights["status_group"].eq("delayed") & flights["label_delay"].isna()
status_non_delay_mask = flights["status_group"].isin(["on_time", "landed", "departed"]) & flights["label_delay"].isna()
flights.loc[status_delay_mask, "label_delay"] = 1
flights.loc[status_non_delay_mask, "label_delay"] = 0

flights[["scheduled_dt", "estimated_dt", "delay_minutes", "status_group", "label_delay"]].head(10)

,scheduled_dt,estimated_dt,delay_minutes,status_group,label_delay
0,2026-04-09 00:10:00,2026-04-09 00:07:00,-3.0,landed,0.0
1,2026-04-09 00:25:00,2026-04-09 00:32:00,7.0,landed,0.0
2,2026-04-09 07:10:00,2026-04-09 07:00:00,-10.0,landed,0.0
3,2026-04-09 07:30:00,2026-04-09 07:16:00,-14.0,landed,0.0
4,2026-04-09 07:55:00,2026-04-09 07:47:00,-8.0,landed,0.0
5,2026-04-09 07:55:00,2026-04-09 07:52:00,-3.0,landed,0.0
6,2026-04-09 08:10:00,2026-04-09 08:13:00,3.0,landed,0.0
7,2026-04-09 08:15:00,2026-04-09 08:41:00,26.0,landed,1.0
8,2026-04-09 08:40:00,2026-04-09 08:34:00,-6.0,landed,0.0
9,2026-04-09 08:55:00,2026-04-09 08:47:00,-8.0,landed,0.0


## Bước 4 - Khóa chuyến bay, loại trùng lặp và tạo snapshot
Bước này tạo flight_key để gom các lần crawl cùng một chuyến bay, loại bản ghi trùng hoàn toàn, sau đó tạo 2 tập:
- flights_current: snapshot mới nhất của mỗi chuyến (phục vụ dashboard realtime)
- flights_training_snapshot: snapshot ưu tiên trước giờ bay 30 phút (phục vụ train model).

In [64]:
key_cols = ["source_airport", "flight_date", "direction", "scheduled_time", "route_airport_std", "flight_number"]

flights = flights.drop_duplicates().copy()
flights = flights.sort_values(["source_airport", "flight_date", "direction", "flight_number", "scheduled_dt", "retrieved_at_vn"])

flights["flight_key"] = flights[key_cols].fillna("").astype(str).agg("|".join, axis=1)

flights_current = (
    flights.sort_values("retrieved_at_vn")
    .groupby("flight_key", as_index=False)
    .tail(1)
    .reset_index(drop=True)
)


def pick_training_snapshot(group: pd.DataFrame) -> pd.DataFrame:
    g = group.sort_values("retrieved_at_vn")
    sched = g["scheduled_dt"].iloc[0]
    if pd.notna(sched):
        cutoff = sched - pd.Timedelta(minutes=TRAINING_SNAPSHOT_LEAD_MINUTES)
        prior = g[g["retrieved_at_vn"] <= cutoff]
        if not prior.empty:
            return prior.tail(1)
    return g.head(1)


flights_training_snapshot = (
    flights.groupby("flight_key", group_keys=False)
    .apply(pick_training_snapshot)
    .reset_index(drop=True)
)

print("Flights after dedup:", flights.shape)
print("Flights current:", flights_current.shape)
print("Flights training snapshot:", flights_training_snapshot.shape)
flights_current.head(3)

Flights after dedup: (2203, 16)
Flights current: (2189, 16)
Flights training snapshot: (2189, 15)


,retrieved_at_vn,flight_date,direction,scheduled_time,estimated_time,route_airport,flight_number,status_raw,source_airport,route_airport_std,status_group,scheduled_dt,estimated_dt,delay_minutes,label_delay,flight_key
0,2026-04-09 23:39:16,2026-04-09,Arrival,15:00,14:51,Phú Quốc (PQC),9G2966,Đã đến,DN,PHU QUOC (PQC),other,2026-04-09 15:00:00,2026-04-09 14:51:00,-9.0,0.0,DN|2026-04-09|Arrival|15:00|PHU QUOC (PQC)|9G2966
1,2026-04-09 23:39:16,2026-04-09,Departure,13:20,13:20,Cần Thơ (VCA),VJ703,27-34,DN,CAN THO (VCA),other,2026-04-09 13:20:00,2026-04-09 13:20:00,0.0,0.0,DN|2026-04-09|Departure|13:20|CAN THO (VCA)|VJ703
2,2026-04-09 23:39:16,2026-04-09,Departure,21:50,21:50,TP. Hồ Chí Minh (SGN),VJ647,27-34,DN,HO CHI MINH (SGN),other,2026-04-09 21:50:00,2026-04-09 21:50:00,0.0,0.0,DN|2026-04-09|Departure|21:50|HO CHI MINH (SGN...


## Bước 5 - Tiền xử lý weather
Bước này chuẩn hóa weather: đổi tên cột, parse giờ báo cáo, map ICAO thành mã sân bay NB/DN/TSN, và chuẩn hóa visibility, gió.

In [56]:
weather = weather_raw.rename(
    columns={
        "ICAO Code": "icao_code",
        "Report Time (UTC)": "report_time_utc",
        "Report Time (VN)": "report_time_vn",
        "Temperature (°C)": "temperature_c",
        "Dew Point (°C)": "dew_point_c",
        "Wind Direction (Degrees)": "wind_direction_deg",
        "Wind Speed (Knots)": "wind_speed_kt",
        "Visibility (Miles)": "visibility_miles",
        "Cloud Cover": "cloud_cover",
        "Raw METAR": "raw_metar",
    }
).copy()

icao_map = {
    "VVNB": "NB",
    "VVDN": "DN",
    "VVTS": "TSN",
}

weather["source_airport"] = weather["icao_code"].map(icao_map)
weather["report_time_vn"] = pd.to_datetime(weather["report_time_vn"], errors="coerce")

weather["visibility_miles"] = (
    weather["visibility_miles"]
    .astype(str)
    .str.replace("+", "", regex=False)
)
weather["visibility_miles"] = pd.to_numeric(weather["visibility_miles"], errors="coerce")

weather["wind_direction_deg"] = weather["wind_direction_deg"].replace({"VRB": np.nan})
weather["wind_direction_deg"] = pd.to_numeric(weather["wind_direction_deg"], errors="coerce")
weather["is_wind_variable"] = weather_raw["Wind Direction (Degrees)"].astype(str).eq("VRB").astype(int)

weather = weather.dropna(subset=["source_airport", "report_time_vn"]).sort_values(["source_airport", "report_time_vn"])
weather = weather.drop_duplicates(subset=["source_airport", "report_time_vn", "raw_metar"], keep="last")

weather.head(3)

,icao_code,report_time_utc,report_time_vn,temperature_c,dew_point_c,wind_direction_deg,wind_speed_kt,visibility_miles,cloud_cover,raw_metar,source_airport,is_wind_variable
143,VVDN,2026-04-08T15:30:00.000Z,2026-04-08 22:30:00,26,25,80.0,4,4.97,FEW@1000ft,METAR VVDN 081530Z 08004KT 8000 FEW010 26/25 Q...,DN,0
140,VVDN,2026-04-08T16:00:00.000Z,2026-04-08 23:00:00,26,24,110.0,3,4.97,"FEW@1000ft, SCT@9000ft",METAR VVDN 081600Z 11003KT 8000 FEW010 SCT090 ...,DN,0
137,VVDN,2026-04-08T16:30:00.000Z,2026-04-08 23:30:00,26,24,NaN,1,4.97,"FEW@1000ft, SCT@6500ft",METAR VVDN 081630Z VRB01KT 8000 FEW010 SCT065 ...,DN,1


## Bước 6 - Ghép weather vào snapshot flight
Bước này dùng merge_asof để ghép bản tin weather gần nhất trước thời điểm retrieved_at_vn cho từng source_airport.

In [65]:
def merge_weather_asof(flight_df: pd.DataFrame, weather_df: pd.DataFrame) -> pd.DataFrame:
    left = flight_df.sort_values(["source_airport", "retrieved_at_vn"]).copy()
    right = weather_df.sort_values(["source_airport", "report_time_vn"]).copy()

    merged_parts = []
    for airport_code, g_left in left.groupby("source_airport"):
        g_right = right[right["source_airport"] == airport_code]
        if g_right.empty:
            g_left = g_left.copy()
            for c in ["temperature_c", "dew_point_c", "wind_direction_deg", "wind_speed_kt", "visibility_miles", "cloud_cover", "is_wind_variable", "raw_metar", "report_time_vn"]:
                g_left[c] = np.nan
            merged_parts.append(g_left)
            continue

        g_merged = pd.merge_asof(
            g_left.sort_values("retrieved_at_vn"),
            g_right.sort_values("report_time_vn"),
            left_on="retrieved_at_vn",
            right_on="report_time_vn",
            direction="backward",
            allow_exact_matches=True,
            suffixes=("", "_wx"),
        )

        if "source_airport_wx" in g_merged.columns:
            g_merged = g_merged.drop(columns=["source_airport_wx"])

        merged_parts.append(g_merged)

    return pd.concat(merged_parts, ignore_index=True)


current_with_weather = merge_weather_asof(flights_current, weather)
train_with_weather = merge_weather_asof(flights_training_snapshot, weather)

print("Current with weather:", current_with_weather.shape)
print("Train with weather:", train_with_weather.shape)
train_with_weather.head(3)

Current with weather: (2189, 27)
Train with weather: (2189, 26)


,retrieved_at_vn,flight_date,direction,scheduled_time,estimated_time,route_airport,flight_number,status_raw,source_airport,route_airport_std,status_group,scheduled_dt,estimated_dt,delay_minutes,label_delay,icao_code,report_time_utc,report_time_vn,temperature_c,dew_point_c,wind_direction_deg,wind_speed_kt,visibility_miles,cloud_cover,raw_metar,is_wind_variable
0,2026-04-09 23:39:16,2026-04-09,Arrival,07:05,06:57,Hà Nội (HAN),QH101,Đúng giờ,DN,HA NOI (HAN),on_time,2026-04-09 07:05:00,2026-04-09 06:57:00,-8.0,0.0,VVDN,2026-04-09T15:00:00.000Z,2026-04-09 22:00:00,27,23,NaN,2,6.0,clear,METAR VVDN 091500Z VRB02KT CAVOK 27/23 Q1010 N...,1
1,2026-04-09 23:39:16,2026-04-09,Departure,13:40,13:40,TP. Hồ Chí Minh (SGN),VN129,3-16,DN,HO CHI MINH (SGN),other,2026-04-09 13:40:00,2026-04-09 13:40:00,0.0,0.0,VVDN,2026-04-09T15:00:00.000Z,2026-04-09 22:00:00,27,23,NaN,2,6.0,clear,METAR VVDN 091500Z VRB02KT CAVOK 27/23 Q1010 N...,1
2,2026-04-09 23:39:16,2026-04-09,Departure,13:20,13:40,Hà Nội (HAN),9G922,35-39,DN,HA NOI (HAN),other,2026-04-09 13:20:00,2026-04-09 13:40:00,20.0,1.0,VVDN,2026-04-09T15:00:00.000Z,2026-04-09 22:00:00,27,23,NaN,2,6.0,clear,METAR VVDN 091500Z VRB02KT CAVOK 27/23 Q1010 N...,1


## Bước 7 - Tạo feature cho model
Bước này tạo các feature thời gian, airline_code, route, và một số feature weather cơ bản. Sau đó lọc ra tập train có nhãn đầy đủ.

In [66]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["scheduled_hour"] = out["scheduled_dt"].dt.hour
    out["scheduled_dayofweek"] = out["scheduled_dt"].dt.dayofweek
    out["scheduled_month"] = out["scheduled_dt"].dt.month

    out["airline_code"] = out["flight_number"].str.extract(r"^([A-Z0-9]{2})", expand=False)
    out["flight_num_only"] = pd.to_numeric(out["flight_number"].str.extract(r"(\d+)", expand=False), errors="coerce")

    out["minutes_to_departure_at_snapshot"] = (out["scheduled_dt"] - out["retrieved_at_vn"]).dt.total_seconds() / 60.0
    out["is_estimated_missing"] = out["estimated_dt"].isna().astype(int)

    out["temp_dew_spread"] = out["temperature_c"] - out["dew_point_c"]
    out["is_low_visibility"] = (out["visibility_miles"] <= 3).astype(float)

    return out


current_features = add_features(current_with_weather)
training_features = add_features(train_with_weather)

training_dataset = training_features[training_features["label_delay"].isin([0, 1])].copy()
training_dataset["label_delay"] = training_dataset["label_delay"].astype(int)

print("Training dataset shape:", training_dataset.shape)
training_dataset[["source_airport", "flight_number", "scheduled_dt", "delay_minutes", "label_delay", "temperature_c", "visibility_miles"]].head(10)

Training dataset shape: (2131, 35)


,source_airport,flight_number,scheduled_dt,delay_minutes,label_delay,temperature_c,visibility_miles
0,DN,QH101,2026-04-09 07:05:00,-8.0,0,27,6.0
1,DN,VN129,2026-04-09 13:40:00,0.0,0,27,6.0
2,DN,9G922,2026-04-09 13:20:00,20.0,1,27,6.0
3,DN,VJ703,2026-04-09 13:20:00,0.0,0,27,6.0
4,DN,VJ510,2026-04-09 13:15:00,0.0,0,27,6.0
5,DN,VN168,2026-04-09 13:00:00,0.0,0,27,6.0
6,DN,VN127,2026-04-09 12:55:00,10.0,0,27,6.0
7,DN,VJ1504,2026-04-09 12:40:00,35.0,1,27,6.0
8,DN,VN123,2026-04-09 11:55:00,10.0,0,27,6.0
9,DN,VN166,2026-04-09 11:35:00,0.0,0,27,6.0


## Bước 8 - Lưu dữ liệu processed
Bước này xuất các tập dữ liệu ra thư mục processed để dùng cho dashboard realtime và model training.

In [67]:
outputs = {
    "flights_clean_all_snapshots.csv": flights,
    "flights_current_snapshot.csv": current_features,
    "flights_training_snapshot.csv": training_features,
    "training_dataset_labeled.csv": training_dataset,
    "weather_clean.csv": weather,
}

for file_name, df_out in outputs.items():
    out_path = processed_dir / file_name
    df_out.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {out_path} | shape={df_out.shape}")

Saved: e:\MyProject\data-monitoring-lab\Data Collection + Processing\data\processed\flights_clean_all_snapshots.csv | shape=(2203, 16)
Saved: e:\MyProject\data-monitoring-lab\Data Collection + Processing\data\processed\flights_current_snapshot.csv | shape=(2189, 36)
Saved: e:\MyProject\data-monitoring-lab\Data Collection + Processing\data\processed\flights_training_snapshot.csv | shape=(2189, 35)
Saved: e:\MyProject\data-monitoring-lab\Data Collection + Processing\data\processed\training_dataset_labeled.csv | shape=(2131, 35)
Saved: e:\MyProject\data-monitoring-lab\Data Collection + Processing\data\processed\weather_clean.csv | shape=(544, 12)


## Bước 9 - Kiểm tra nhanh kết quả
Bước này thống kê nhanh tỷ lệ delay và số mẫu theo sân bay để xác nhận dữ liệu train có hợp lý.

In [68]:
summary_by_airport = training_dataset.groupby("source_airport").agg(
    samples=("label_delay", "count"),
    delay_rate=("label_delay", "mean"),
    avg_delay_minutes=("delay_minutes", "mean"),
)

summary_by_airport["delay_rate"] = (summary_by_airport["delay_rate"] * 100).round(2)
summary_by_airport["avg_delay_minutes"] = summary_by_airport["avg_delay_minutes"].round(2)

print("Tong so mau train:", len(training_dataset))
summary_by_airport

Tong so mau train: 2131


,samples,delay_rate,avg_delay_minutes
source_airport,,,
DN,689,14.22,3.99
NB,1338,13.90,3.74
TSN,104,32.69,5.94
